# Evaluator Agent — Code Guide

**Owner:** Sabina Rudolph  
**Target file:** `agents/sabina_evaluator.py`  

> This notebook is documentation only — do not run it. It mirrors the actual agent code block by block and explains what each part does in plain English to make it easier for us to understand the project and our code inside out.

## What this agent does

This notebook documents the Evaluator Agent, the third handoff of the stock-move prediction pipeline. It reads the Classifier Agent's `predictions_test.csv` and generated `classifier.py`, scores the requested held-out split (`val` during retunes, `test` for the final report), reviews the generated code, and writes a contract-faithful `evaluation_report.json` back to the Manager Agent.

In simpler words: the Classifier Agent gives me the model's guesses, and my agent grades them. It checks how often the model was right, which label was weakest, and whether the generated classifier code has anything obvious the Manager Agent should know about.

The evaluator's main job is to produce the metrics and proposal the Manager Agent needs for the retune decision. My evaluator agent calls local Ollama when enabled. The LLM reviews a small sample of misclassified headlines and improves only the human-readable `reason` and `code_notes` fields.

The important design choice is that the control fields stay deterministic:

- `metrics`
- `recommended_action`
- `focus_labels`
- `suggested_params`

So the LLM can add useful judgement, but it cannot secretly change the retune gate.


## 1. Imports and constants

These are the standard libraries and constants the evaluator needs:

- **`json`** — write the Manager Agent's JSON report
- **`os`** — build output paths and read environment variables
- **`re`** — inspect simple assignments inside the Classifier Agent's generated `classifier.py`
- **`urllib.request`** — call local Ollama without adding another dependency
- **`TypedDict`** — document the state passed through LangGraph
- **`Agent`** — the shared base class so all agents expose the same `.run()` style interface
- **`LABELS` / `PREDICTION_COLUMNS` / contract readers** — shared Handoff 2 rules from `agents/contracts.py`

The evaluator still owns its scoring and proposal constants, because those are evaluator policy. The cross-agent file shape lives in `agents/contracts.py`, so Sabina, Nadi, Jack, and Freddi all validate and shape handoff rows from the same contract source. The only environment-configurable values are the Ollama settings because those genuinely vary by machine.

In [ ]:
import json
import os
import re
import urllib.request
from typing import Callable, TypedDict

try:
    from agents.base import Agent
    from agents.contracts import (
        LABELS,
        PREDICTION_COLUMNS,
        read_prediction_rows,
        validate_prediction_rows,
    )
except ModuleNotFoundError:
    from base import Agent
    from contracts import (
        LABELS,
        PREDICTION_COLUMNS,
        read_prediction_rows,
        validate_prediction_rows,
    )

OUTPUT_DIR = "outputs"
TARGET_ACCURACY = 0.60
DEFAULT_RETUNE_THRESHOLD = 0.50
THRESHOLD_STEP = 0.05
THRESHOLD_FLOOR = 0.20
DEFAULT_MAX_LENGTH = 128
FOCUS_MARGIN = 0.05

# Read once at import time so tests and demos get stable evaluator behavior.
USE_OLLAMA = os.getenv("EVALUATOR_USE_OLLAMA", "false").lower() == "true"
OLLAMA_URL = os.getenv("OLLAMA_URL", "http://localhost:11434/api/generate")
OLLAMA_MODEL = os.getenv("EVALUATOR_OLLAMA_MODEL", "llama3.1")
OLLAMA_TIMEOUT_SECONDS = 30
LLM_TEMPERATURE = 0.2
MISCLASSIFIED_SAMPLE_SIZE = 25

PROPOSAL_FIELDS = {
    "recommended_action", "reason", "focus_labels", "suggested_params", "code_notes",
}

## 2. Evaluator state

This is the small state dictionary that LangGraph passes through my evaluator graph.

It starts with file paths:

- `predictions_path` — where the Classifier Agent wrote `predictions_test.csv`
- `classifier_code_path` — where the Classifier Agent wrote `classifier.py`
- `output_path` — where I should write `evaluation_report.json`

Then the graph adds the loaded data, the code text, and finally the finished report.

I kept this state intentionally small because the Evaluator Agent's job is just one handoff: read the Classifier Agent's outputs and write the Manager Agent's input.

In [ ]:
class EvaluatorState(TypedDict, total=False):
    predictions_path: str
    classifier_code_path: str
    output_path: str
    predictions: list[dict]
    code_text: str
    code_notes: str
    report: dict

## 3. Reading and validating the Classifier Agent's files

Before calculating anything, the evaluator checks that the Classifier Agent's CSV matches the data contract. This is important because otherwise the accuracy number could look valid while being based on the wrong columns or mixed train/test data.

The validation rules live in `agents/contracts.py`. Sabina calls those helpers instead of re-implementing them locally, because the same Handoff 2 rules are also used by Nadi's generated-code guardrail and Jack's final output path. Sabina later chooses which split to score with `eval_split`.

The validation checks:

- the columns are exactly the Handoff 2 columns
- the file is not empty
- every row is from a held-out split (`val` or `test`), never `train`
- labels and predictions are only `up`, `down`, or `neutral`
- probabilities and confidence are in the 0-1 range
- `prob_up + prob_down + prob_neutral` is approximately 1
- `confidence` equals the highest probability

This is a bit strict on purpose. If the classifier output is malformed, it is better to fail early than hand the Manager Agent a misleading report. It is basically the boring checking part, but it matters because everything after this depends on the file being correct.

In [ ]:
def _read_predictions(path: str) -> list[dict]:
    # Read classifier predictions and enforce the exact Handoff 2 columns.
    return read_prediction_rows(path)


def _read_code(path: str) -> str:
    with open(path, encoding="utf-8") as f:
        return f.read()


def validate_predictions(rows: list[dict]) -> None:
    # Check the Handoff 2 fields needed before scoring.
    validate_prediction_rows(rows)

## 4. Computing the metrics

This is the report card part of the evaluator.

The main metric is **accuracy**, because this is a three-class classification task (`up`, `down`, `neutral`). I also calculate per-class accuracy and per-class support so the Manager Agent can tell whether one class is weak or simply absent from the selected split. That matters for retuning because the Classifier Agent can focus on weak labels later without mistaking zero-support labels for class collapse.

The output of this function becomes the top-level metric section of `evaluation_report.json`: `accuracy`, `below_threshold`, `class_accuracy`, `class_support`, `misclassified_count`, and `misclassified_ids`.

In [ ]:
def compute_metrics(rows: list[dict]) -> dict:
    # Compute overall and per-class classification accuracy/support.
    total = len(rows)
    wrong = [row for row in rows if row["label"] != row["predicted_label"]]
    class_accuracy = {}
    class_support = {}

    for label_name in LABELS:
        class_rows = [row for row in rows if row["label"] == label_name]
        correct = sum(row["predicted_label"] == label_name for row in class_rows)
        class_support[label_name] = len(class_rows)
        class_accuracy[label_name] = (
            round(correct / len(class_rows), 2) if class_rows else 0.0
        )

    accuracy = round((total - len(wrong)) / total, 2)
    return {
        "accuracy": accuracy,
        "below_threshold": accuracy < TARGET_ACCURACY,
        "class_accuracy": class_accuracy,
        "class_support": class_support,
        "misclassified_count": len(wrong),
        "misclassified_ids": [row["article_id"] for row in wrong],
    }

## 5. Helpers — reading simple settings from `classifier.py`

The Classifier Agent's classifier is generated code, but there are a few useful constants inside it, like `THRESHOLD`, `MAX_LENGTH`, `MODEL`, and `MODEL_DIR`.

The evaluator does not execute the classifier again. It only reads the source text and looks for simple assignment lines. This is enough to explain what the classifier used and to suggest the next retune values.

For example, if the classifier currently has `THRESHOLD = 0.50`, the Evaluator Agent can suggest `0.45` for the next retune.

In [ ]:
def _find_assignment(code_text: str, name: str) -> str | None:
    match = re.search(rf"^\s*{re.escape(name)}\s*=\s*([^\n#]+)", code_text, re.M)
    return match.group(1).strip() if match else None


def _float_assignment(code_text: str, name: str, default: float) -> float:
    value = _find_assignment(code_text, name)
    if value is None:
        return default
    try:
        return float(value.strip("\"'"))
    except ValueError:
        return default


def _int_assignment(code_text: str, name: str, default: int) -> int:
    value = _find_assignment(code_text, name)
    if value is None:
        return default
    try:
        return int(float(value.strip("\"'")))
    except ValueError:
        return default

## 6. Helper — finding the weakest labels

Originally it would have been easy to only pick the exact weakest class. But in real data, class accuracies almost never tie exactly.

So the final version uses a small margin: every class within `0.05` of the weakest class is included in `focus_labels`.

Example:

- `up = 0.30`
- `down = 0.28`
- `neutral = 0.45`

The exact weakest is `down`, but `up` is basically also weak. With `FOCUS_MARGIN = 0.05`, both `down` and `up` are included. That gives the Classifier Agent a better signal for retuning.

In [ ]:
def _weakest_labels(class_accuracy: dict) -> list[str]:
    weakest_score = min(class_accuracy.values())
    return [
        label_name
        for label_name, score in class_accuracy.items()
        if score <= weakest_score + FOCUS_MARGIN
    ]

## 7. Static classifier review

This function writes short notes about the generated classifier code. It is intentionally simple and deterministic.

Right now it looks for two useful signals:

- whether the threshold is hardcoded
- whether the neutral class is one of the weakest labels

These notes later become `code_notes`. If Ollama is enabled, the LLM can rewrite these notes in a more helpful way, but the static version always exists as a fallback.

In [ ]:
def review_classifier_code(code_text: str, class_accuracy: dict) -> str:
    # Return concise static observations from the Classifier Agent's generated classifier.py.
    notes = []

    threshold = _find_assignment(code_text, "THRESHOLD")
    if threshold is not None:
        notes.append(f"threshold hardcoded at {threshold} in classifier.py")

    weakest_labels = _weakest_labels(class_accuracy)
    if "neutral" in weakest_labels:
        notes.append("the neutral band (+/-1%) may be too narrow for the neutral class")

    return "; ".join(notes)

## 8. Suggesting retune parameters

This is one of the parts where the evaluator must stay deterministic.

If accuracy is below the target, the Evaluator Agent recommends `retune`. The suggested threshold steps down by `0.05` from the classifier's current threshold, but never below `0.20`. That floor matches the Manager Agent's retune schedule.

`max_length` is read from the classifier if possible, otherwise it defaults to `128`.

The LLM is not allowed to invent these values because the Manager Agent and Classifier Agent depend on them being stable.

In [ ]:
def _suggest_retune_params(code_text: str) -> dict:
    # Suggest the next deterministic retune step without asking the LLM.
    current_threshold = _float_assignment(
        code_text, "THRESHOLD", DEFAULT_RETUNE_THRESHOLD
    )
    next_threshold = max(THRESHOLD_FLOOR, current_threshold - THRESHOLD_STEP)
    max_length = _int_assignment(code_text, "MAX_LENGTH", DEFAULT_MAX_LENGTH)

    return {
        "threshold": round(next_threshold, 2),
        "max_length": max_length,
    }

## 9. Building the deterministic base proposal

The `proposal` object is the Evaluator Agent's recommendation to the Manager Agent. The Manager Agent still owns the final decision.

This function builds the first version of the proposal completely without the LLM:

- if accuracy is below `0.60` -> recommend `retune`
- if accuracy clears `0.60` -> recommend `proceed`
- include the weakest / near-weakest labels
- include suggested params only for retune
- include a plain English reason and code notes

This is the version we fall back to if Ollama is off, unavailable, or returns invalid JSON.

In [ ]:
def make_base_proposal(metrics: dict, code_text: str, code_notes: str) -> dict:
    # Create the contract proposal with deterministic action and params.
    weakest_score = min(metrics["class_accuracy"].values())
    focus_labels = _weakest_labels(metrics["class_accuracy"])

    if metrics["below_threshold"]:
        reason = (
            f"accuracy {metrics['accuracy']:.2f} below target "
            f"{TARGET_ACCURACY:.2f}; {', '.join(focus_labels)} class weakest"
        )
        return {
            "recommended_action": "retune",
            "reason": reason,
            "focus_labels": focus_labels,
            "suggested_params": _suggest_retune_params(code_text),
            "code_notes": code_notes,
        }

    reason = (
        f"accuracy {metrics['accuracy']:.2f} clears the {TARGET_ACCURACY:.2f} "
        f"target; {', '.join(focus_labels)} class is weakest "
        f"({weakest_score:.2f}) but the iteration budget favours proceeding"
    )
    return {
        "recommended_action": "proceed",
        "reason": reason,
        "focus_labels": focus_labels,
        "suggested_params": {},
        "code_notes": code_notes,
    }

## 10. Validating the proposal

This is the safety layer. Even though the base proposal is deterministic, I still validate it, and I also validate the final proposal after any LLM text is applied.

The validator makes sure:

- all required fields exist
- no extra fields sneak in
- `recommended_action` is only `retune` or `proceed`
- `recommended_action` matches the accuracy gate
- focus labels are real labels
- `suggested_params` is a dictionary
- `suggested_params` only appear when the action is `retune`
- retune params include numeric `threshold` and integer `max_length`
- `reason` and `code_notes` have the expected text types

This is why the LLM can be useful without becoming dangerous. It can write text, but it cannot take over the pipeline contract.


In [ ]:
def validate_proposal(proposal: dict, metrics: dict) -> dict:
    """Validate the Manager Agent's proposal object before it can enter evaluation_report.json."""
    if not isinstance(proposal, dict):
        raise ValueError("LLM proposal must be a JSON object")

    missing = PROPOSAL_FIELDS - set(proposal)
    extra = set(proposal) - PROPOSAL_FIELDS
    if missing:
        raise ValueError(f"LLM proposal missing fields: {sorted(missing)}")
    if extra:
        raise ValueError(f"LLM proposal has unexpected fields: {sorted(extra)}")

    action = proposal["recommended_action"]
    if action not in {"retune", "proceed"}:
        raise ValueError("recommended_action must be retune or proceed")

    expected_action = "retune" if metrics["below_threshold"] else "proceed"
    if action != expected_action:
        raise ValueError(
            "recommended_action conflicts with deterministic threshold gate: "
            f"expected {expected_action}, got {action}"
        )

    focus_labels = proposal["focus_labels"]
    if not isinstance(focus_labels, list) or not focus_labels:
        raise ValueError("focus_labels must be a non-empty list")
    invalid_labels = [label for label in focus_labels if label not in LABELS]
    if invalid_labels:
        raise ValueError(f"focus_labels contains invalid labels: {invalid_labels}")

    if not isinstance(proposal["suggested_params"], dict):
        raise ValueError("suggested_params must be an object")
    if action == "proceed" and proposal["suggested_params"] != {}:
        raise ValueError("proceed proposals must not carry suggested_params")
    if action == "retune":
        params = proposal["suggested_params"]
        if not {"threshold", "max_length"} <= set(params):
            raise ValueError("retune proposals need threshold and max_length")
        if not isinstance(params["threshold"], (int, float)):
            raise ValueError("threshold must be numeric")
        if not isinstance(params["max_length"], int):
            raise ValueError("max_length must be an integer")
    if not isinstance(proposal["reason"], str) or not proposal["reason"].strip():
        raise ValueError("reason must be a non-empty string")
    if not isinstance(proposal["code_notes"], str):
        raise ValueError("code_notes must be a string")

    return proposal

## 11. Preparing a small LLM prompt

The evaluator agent does not send the whole classifier source or every misclassified id into the prompt. That would be too large and not very helpful.

Instead, the LLM gets:

- compact metrics
- a small classifier summary (`threshold`, `max_length`, `MODEL`, `MODEL_DIR`)
- up to 25 misclassified examples
- the deterministic proposal

The misclassified sample includes the headline, true label, predicted label, confidence, and probabilities. This gives the LLM enough context to describe a real failure pattern, for example if `up` headlines are often being predicted as `neutral`.

So the LLM is not just rewriting a sentence. It gets a small set of actual mistakes and can say what kind of mistake the model seems to make. The prompt still says very clearly that the LLM may only return two fields: `reason` and `code_notes`.


In [ ]:
def _prompt_metrics(metrics: dict) -> dict:
    # Small metrics view for the LLM; opaque row ids stay out of the prompt.
    return {
        "accuracy": metrics["accuracy"],
        "below_threshold": metrics["below_threshold"],
        "class_accuracy": metrics["class_accuracy"],
        "misclassified_count": metrics["misclassified_count"],
    }


def _misclassified_sample(rows: list[dict], limit: int = MISCLASSIFIED_SAMPLE_SIZE) -> list[dict]:
    # Small failure sample for LLM pattern analysis; row ids stay out.
    sample = []
    for row in rows:
        if row["label"] == row["predicted_label"]:
            continue
        sample.append({
            "headline": row["article_title"],
            "true_label": row["label"],
            "predicted_label": row["predicted_label"],
            "confidence": float(row["confidence"]),
            "prob_up": float(row["prob_up"]),
            "prob_down": float(row["prob_down"]),
            "prob_neutral": float(row["prob_neutral"]),
        })
        if len(sample) >= limit:
            break
    return sample


def _classifier_summary_for_prompt(code_text: str) -> dict:
    # Summarise source signals instead of sending the whole generated file.
    return {
        "threshold": _find_assignment(code_text, "THRESHOLD"),
        "max_length": _find_assignment(code_text, "MAX_LENGTH"),
        "model": _find_assignment(code_text, "MODEL"),
        "model_dir": _find_assignment(code_text, "MODEL_DIR"),
        "uses_finbert": "finbert" in code_text.lower(),
        "maps_sentiment_to_label": "predicted_label" in code_text,
    }

## 12. Calling Ollama and falling back safely

This is the agentic part, but with guardrails.

If `EVALUATOR_USE_OLLAMA=true`, my evaluator agent calls local Ollama. The LLM response must be JSON with exactly:

```json
{"reason": "...", "code_notes": "..."}
```

If the response is invalid, times out, or tries to return extra fields, the evaluator prints a small log line and uses the deterministic base proposal. This means the pipeline can still run offline.

This was important for the course demo because not everyone has the same local model setup.

In [ ]:
def _build_llm_prompt(metrics: dict, code_text: str, base_proposal: dict, failure_sample: list[dict]) -> str:
    # Prompt the LLM for judgement text only; control fields are deterministic.
    payload = {
        "metrics": _prompt_metrics(metrics),
        "classifier_summary": _classifier_summary_for_prompt(code_text),
        "misclassified_sample": failure_sample,
        "deterministic_proposal": base_proposal,
    }
    return (
        "You are the evaluator agent in a multi-agent stock-move prediction pipeline.\n\n"
        "Review the deterministic metrics and classifier summary. The action, "
        "focus labels, and suggested params are already fixed by code and must "
        "not be changed by you.\n\n"
        "Return ONLY valid JSON with exactly these two string fields:\n"
        '{"reason": "...", "code_notes": "..."}\n\n'
        "Ground your reason in accuracy, target, weakest labels, and any useful "
        "classifier observation. Use the misclassified sample to describe the "
        "failure pattern when possible. Do not include markdown or extra keys.\n\n"
        f"INPUT:\n{json.dumps(payload, indent=2)}"
    )


def _ollama_generate(prompt: str) -> str:
    # Call local Ollama when explicitly enabled; no API key is needed.
    body = json.dumps({
        "model": OLLAMA_MODEL,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": LLM_TEMPERATURE},
    }).encode("utf-8")

    request = urllib.request.Request(
        OLLAMA_URL,
        data=body,
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(request, timeout=OLLAMA_TIMEOUT_SECONDS) as response:
        payload = json.loads(response.read().decode("utf-8"))
    return payload.get("response", "")

## 12b. Parsing and validating the LLM response

Ollama sometimes returns clean JSON, but sometimes a model wraps JSON in Markdown fences or adds a little text around it. `_extract_json_object` is a small tolerance layer for that.

After parsing, `_validate_llm_review` is strict again. It only accepts exactly two string fields:

- `reason`
- `code_notes`

If the model tries to return `recommended_action`, `focus_labels`, or `suggested_params`, the response is rejected and the deterministic fallback is used. This was one of the most important safeguards in the final version.

In [ ]:
def _extract_json_object(text: str) -> dict:
    # Parse plain JSON or a fenced JSON object returned by an LLM.
    stripped = text.strip()
    if stripped.startswith("```"):
        stripped = re.sub(r"^```(?:json)?\s*", "", stripped, flags=re.I)
        stripped = re.sub(r"\s*```$", "", stripped)
    try:
        return json.loads(stripped)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", stripped, re.S)
        if not match:
            raise
        return json.loads(match.group(0))


def _validate_llm_review(review: dict) -> dict:
    # Accept only the fields the LLM is allowed to write.
    if not isinstance(review, dict):
        raise ValueError("LLM review must be a JSON object")
    if set(review) != {"reason", "code_notes"}:
        raise ValueError("LLM review may only contain reason and code_notes")
    if not isinstance(review["reason"], str) or not review["reason"].strip():
        raise ValueError("LLM reason must be a non-empty string")
    if not isinstance(review["code_notes"], str):
        raise ValueError("LLM code_notes must be a string")
    return {
        "reason": review["reason"].strip(),
        "code_notes": review["code_notes"].strip(),
    }

## 13. Applying the LLM review

This function is where the base proposal and LLM output meet.

The logic is:

1. If no LLM is configured, return the base proposal.
2. Build the prompt.
3. Call either the injected test LLM function or real Ollama.
4. Parse the JSON response.
5. Accept only `reason` and `code_notes`.
6. Merge those two text fields into the base proposal.
7. Validate the final proposal again.

So even when Ollama is on, the final action and retune params still come from deterministic code.

In [ ]:
def apply_llm_review(
    metrics: dict,
    code_text: str,
    base_proposal: dict,
    failure_sample: list[dict],
    llm_fn: Callable[[str], str] | None = None,
) -> dict:
    # Let an LLM improve reason/code_notes without changing control fields.
    if llm_fn is None and not USE_OLLAMA:
        return base_proposal

    prompt = _build_llm_prompt(metrics, code_text, base_proposal, failure_sample)
    try:
        response = (llm_fn or _ollama_generate)(prompt)
        review = _validate_llm_review(_extract_json_object(response))
    except (ValueError, json.JSONDecodeError, TimeoutError, OSError) as error:
        print(f"[sabina] LLM review failed ({error}); using deterministic fallback.")
        return base_proposal

    proposal = {
        **base_proposal,
        "reason": review["reason"],
        "code_notes": review["code_notes"],
    }
    return validate_proposal(proposal, metrics)

## 14. Building the final report

`build_report` is the core of the evaluator. It connects all helper functions in the actual order the agent uses them:

1. validate the Classifier Agent's predictions
2. compute metrics
3. statically review the classifier code
4. build the deterministic base proposal
5. collect a small misclassified sample
6. optionally let the LLM improve the text fields
7. return the full `evaluation_report.json` object

This function is also easy to test because it accepts rows and code text directly. The tests can pass a fake `llm_fn`, so they do not need live Ollama or network calls.

In [ ]:
def build_report(
    rows: list[dict],
    code_text: str,
    llm_fn: Callable[[str], str] | None = None,
) -> dict:
    validate_predictions(rows)
    metrics = compute_metrics(rows)
    code_notes = review_classifier_code(code_text, metrics["class_accuracy"])

    base_proposal = validate_proposal(
        make_base_proposal(metrics, code_text, code_notes),
        metrics,
    )

    failure_sample = _misclassified_sample(rows)
    proposal = apply_llm_review(
        metrics,
        code_text,
        base_proposal,
        failure_sample,
        llm_fn=llm_fn,
    )

    return {**metrics, "proposal": proposal}

## 15. LangGraph nodes

The evaluator graph has only three nodes, because the Evaluator Agent's job is a straight handoff:

```text
load_inputs -> evaluate -> write_report
```

- `load_inputs` reads the CSV and classifier code
- `evaluate` builds the report
- `write_report` writes `evaluation_report.json`

I kept the graph simple on purpose. The retune loop itself belongs to the Manager Agent, not to the Evaluator Agent.

In [ ]:
def _write_json(path: str, obj: dict) -> None:
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)


def load_inputs(state: EvaluatorState) -> dict:
    return {
        "predictions": _read_predictions(state["predictions_path"]),
        "code_text": _read_code(state["classifier_code_path"]),
    }


def evaluate(state: EvaluatorState) -> dict:
    return {"report": build_report(state["predictions"], state["code_text"])}


def write_report(state: EvaluatorState) -> dict:
    output_path = state.get("output_path") or os.path.join(OUTPUT_DIR, "evaluation_report.json")
    _write_json(output_path, state["report"])
    return {"output_path": output_path}

## 16. EvaluatorAgent class and test entry point

This class wraps the graph in the same style as the other team agents. The pipeline can call `EvaluatorAgent().run(...)` without needing to know the internal function names.

The `if __name__ == "__main__"` block is just for local testing. It runs the evaluator on the mock files and prints the output path. In the real pipeline, this agent is called through the graph instead.

In [ ]:
def build_graph(checkpointer):
    from langgraph.graph import StateGraph, START, END

    builder = StateGraph(EvaluatorState)
    builder.add_node("load_inputs", load_inputs)
    builder.add_node("evaluate", evaluate)
    builder.add_node("write_report", write_report)
    builder.add_edge(START, "load_inputs")
    builder.add_edge("load_inputs", "evaluate")
    builder.add_edge("evaluate", "write_report")
    builder.add_edge("write_report", END)
    return builder.compile(checkpointer=checkpointer)


class EvaluatorAgent(Agent):
    # Evaluator behind the shared .run() interface.
    def __init__(self, *, output_dir=OUTPUT_DIR, checkpointer=None, thread_id="evaluator"):
        self._output_dir = output_dir
        super().__init__(checkpointer=checkpointer, thread_id=thread_id)

    def build_graph(self, checkpointer):
        return build_graph(checkpointer)

    def run(self, predictions: str, classifier_code: str) -> dict:
        output_path = os.path.join(self._output_dir, "evaluation_report.json")
        return self._invoke({
            "predictions_path": predictions,
            "classifier_code_path": classifier_code,
            "output_path": output_path,
        })


if __name__ == "__main__":
    agent = EvaluatorAgent()
    state = agent.run(
        predictions="mock_data/predictions_test.csv",
        classifier_code="mock_data/classifier.py",
    )
    print("Output file:", state["output_path"])

## 17. How I would explain the final design

My main goal was to make my evaluator agent useful but not risky. The evaluator has to be trustworthy because the Manager Agent's loop depends on it.

So the final version separates the responsibilities:

- deterministic code calculates the metrics
- deterministic code decides whether the report is below threshold
- deterministic code chooses `focus_labels` and `suggested_params`
- the LLM only improves the human-readable reasoning

This means Ollama can add real signal by looking at a small sample of misclassified headlines, but the pipeline still works offline and still follows the data contract.

For the demo, Ollama can be enabled like this:

```bash
EVALUATOR_USE_OLLAMA=true EVALUATOR_OLLAMA_MODEL=llama3.2 uv run main.py
```

If Ollama is not running, the Evaluator Agent prints a fallback message and still writes a valid `evaluation_report.json`.

The short version is: my code grades the Classifier Agent's classifier, suggests what to do next, and lets the LLM help with the explanation without giving it control over the decision.
